# Testing the `AdvisorWorkFlow _v1.3` n8n Workflow (flattened webhook contract)

This notebook targets the webhook's **actual current contract**, confirmed
against the workflow's own pinned example data.

### Why flattened, not nested?
The `Webhook` node feeds into `aggregate_webhook_input`:

```js
return { ...$input.first().json.body, inputPath: 'webhook' };
```

This just spreads the POST body's top-level keys through — it does **not**
replicate the flattening that `parsing input for extractor` performs for
the `Start` (sub-workflow) entry point. But the `Debt Solution Extractor`
agent's prompt reads flat fields directly (`$json.accText`,
`$json.consultAcc`, `$json.DebtSituation`, `$json.maxPayment`, etc.) — not
`$json.conversationDesc.consultAcc`. So whatever you POST must already be
flat, or those prompt variables come through empty.

> **Note:** this is arguably a gap in the workflow — the `Start` and
> `Webhook` entry points aren't equivalent. Until that's fixed on the n8n
> side, this notebook targets the contract the workflow expects *today*.

### Required payload fields
| Field | Meaning |
|---|---|
| `consultAcc` | comma-separated account numbers, or `""` |
| `DebtSituation` | one of 5 situation enums, or `null` |
| `maxPayment`, `maxTerm` | numbers or `null` |
| `refPlanID`, `maxPaymentY2`, `maxPaymentY3` | optional, `null` if unset |
| `narrative` | running English summary |
| `accText` | **JSON-stringified** array of account summaries |
| `offerText` | **JSON-stringified** array of offered plans |
| `accInfotoExtr` | the same account summaries, as a real array (not stringified) |
| `LastAImessage` | **JSON-stringified string** — i.e. the bot's previous reply, double-quoted |
| `userMessage` | latest customer message |

### What the webhook returns
Only the raw extractor output:
```json
{
  "consultAcc": "...", "maxPayment": 0, "maxTerm": 0,
  "DebtSituation": "...", "refPlanID": null,
  "maxPaymentY2": null, "maxPaymentY3": null, "reConfirmMessage": ""
}
```
(The offer-engine HTTP call and reply templates only run on the `Start`
sub-workflow path, not reachable here.)

### URL note
- Test URL (n8n editor "Listen for test event" open): `{base}/webhook-test/{path}`
- Production URL (workflow Active, which this one is): `{base}/webhook/{path}`

Set `N8N_BASE_URL` below to your actual n8n instance.

In [1]:
import json
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Optional

import pandas as pd
import requests

## 1. Configuration

In [2]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "b7607735-bac3-4965-8c68-2ed0ef38aec4"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

'https://alphamakeathon-automation.arisetech.dev/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4'

## 2. Account fixtures + flattening helpers\n\nMirrors the exact logic inside `parsing input for extractor`, in Python.

In [3]:
BASE_ACC_INFO: list[dict[str, Any]] = [
    {
        "accNo": "1xxxxxxx0974",
        "port": "สินเชื่อส่วนบุคคล",
        "creditLimit": 30000,
        "os": 19913.56,
        "installment": 1700,
        "remainTerm": 14,
    },
    {
        "accNo": "1xxxxxxx0975",
        "port": "สินเชื่อส่วนบุคคล",
        "creditLimit": 80000,
        "os": 72742.01,
        "installment": 3500,
        "remainTerm": 27,
    },
]


def transform_acc_info(acc_info: list[dict]) -> list[dict]:
    """Mirrors the `accUsed` mapping inside `parsing input for extractor`."""
    return [
        {
            "account_number": acc.get("accNo"),
            "port": acc.get("port"),
            "creditlimit": acc.get("creditLimit"),
            "os": acc.get("os"),
            "installment": acc.get("installment"),
            "remaining_term": acc.get("remainTerm"),
        }
        for acc in acc_info
    ]


def stringify_last_ai_message(content: str) -> str:
    """Mirrors `JSON.stringify($input.first().json.LastAImessage.content)`."""
    return json.dumps(content, ensure_ascii=False)

## 3. Payload model + caller function

In [4]:
@dataclass
class AdvisorWebhookPayload:
    userMessage: str
    consultAcc: str = ""
    DebtSituation: Optional[str] = None
    maxPayment: Optional[float] = None
    maxTerm: Optional[int] = None
    refPlanID: Optional[str] = None
    maxPaymentY2: Optional[float] = None
    maxPaymentY3: Optional[float] = None
    narrative: str = ""
    accInfo: list = field(default_factory=lambda: deepcopy(BASE_ACC_INFO))
    offerSoln: list = field(default_factory=list)  # e.g. [{"planId": "P1", "term": 36, "installment": 5000}]
    lastAIMessageContent: str = ""  # plain text of the bot's previous reply (or "" if none)

    def to_dict(self) -> dict:
        acc_used = transform_acc_info(self.accInfo)
        return {
            "consultAcc": self.consultAcc,
            "DebtSituation": self.DebtSituation,
            "maxPayment": self.maxPayment,
            "maxTerm": self.maxTerm,
            "refPlanID": self.refPlanID,
            "maxPaymentY2": self.maxPaymentY2,
            "maxPaymentY3": self.maxPaymentY3,
            "narrative": self.narrative,
            "accText": json.dumps(acc_used, ensure_ascii=False, separators=(",", ":")),
            "offerText": json.dumps(self.offerSoln, ensure_ascii=False, separators=(",", ":")),
            "accInfotoExtr": acc_used,
            "LastAImessage": stringify_last_ai_message(self.lastAIMessageContent),
            "userMessage": self.userMessage,
        }


def call_advisor(payload: AdvisorWebhookPayload, timeout: int = 60) -> dict:
    """POST a payload to the Advisor webhook and return the parsed JSON response."""
    url = get_webhook_url()
    response = requests.post(url, json=payload.to_dict(), timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Sanity check — does our flattening match the workflow's own pinned example?

This confirms the `AdvisorWebhookPayload` builder reproduces exactly what
`parsing input for extractor` would have produced, so we know the test
payloads below are well-formed.

In [5]:
REFERENCE_PAYLOAD = {
    "consultAcc": "",
    "DebtSituation": "DebtBurden",
    "maxPayment": 5200,
    "maxTerm": 360,
    "refPlanID": None,
    "maxPaymentY2": None,
    "maxPaymentY3": None,
    "narrative": (
        "The customer specifies their payment ability, stating they can only "
        "afford 3,000 THB per month for their two personal loan accounts "
        "combined. They are asking for a new repayment plan based on this "
        "amount. Routing to the advisor to create a personalized solution."
    ),
    "accText": (
        '[{"account_number":"1xxxxxxx0974","port":"สินเชื่อส่วนบุคคล","creditlimit":30000,'
        '"os":19913.56,"installment":1700,"remaining_term":14},{"account_number":"1xxxxxxx0975",'
        '"port":"สินเชื่อส่วนบุคคล","creditlimit":80000,"os":72742.01,"installment":3500,"remaining_term":27}]'
    ),
    "offerText": "[]",
    "accInfotoExtr": transform_acc_info(BASE_ACC_INFO),
    "LastAImessage": None,  # filled in below
    "userMessage": "ทั้งสองบัญชีนี้อยากผ่อนแค่เดือนละ 3000 ได้ไหม",
}

BASELINE_LAST_AI_CONTENT = (
    "น้องฟิน ยินดีให้บริการคำปรึกษาแก้ปัญหาหนี้ค่ะ เพื่อให้ระบบสามารถวิเคราะห์แนวทางการแก้ปัญหาหนี้ได้ถูกต้อง "
    "กรุณาระบุความสามารถในการชำระสินเชื่อและระบุบัญชีที่ลูกค้าต้องการ จากบัญชีสินเชื่อธนาคารกรุงไทยที่พบบนระบบ ดังนี้ค่ะ\n\n"
    "บัญชีที่ 1\nเลขที่บัญชี: 1xxxxxxx0974\nประเภทสินเชื่อ: สินเชื่อส่วนบุคคล\nยอดชำระคงเหลือ: 19,913.56 บาท\n"
    "อัตราผ่อนชำระ: 1,700 บาท/เดือน\nระยะเวลาคงเหลือ: 14 เดือน\n\n"
    "บัญชีที่ 2\nเลขที่บัญชี: 1xxxxxxx0975\nประเภทสินเชื่อ: สินเชื่อส่วนบุคคล\nยอดชำระคงเหลือ: 72,742.01 บาท\n"
    "อัตราผ่อนชำระ: 3,500 บาท/เดือน\nระยะเวลาคงเหลือ: 27 เดือน"
)
REFERENCE_PAYLOAD["LastAImessage"] = stringify_last_ai_message(BASELINE_LAST_AI_CONTENT)

baseline_payload = AdvisorWebhookPayload(
    userMessage="ทั้งสองบัญชีนี้อยากผ่อนแค่เดือนละ 3000 ได้ไหม",
    consultAcc="",
    DebtSituation="DebtBurden",
    maxPayment=5200,
    maxTerm=360,
    narrative=REFERENCE_PAYLOAD["narrative"],
    lastAIMessageContent=BASELINE_LAST_AI_CONTENT,
)

assert baseline_payload.to_dict() == REFERENCE_PAYLOAD, "Mismatch vs the workflow's pinned example!"
print("✅ Builder output matches the workflow's pinned example exactly.")

✅ Builder output matches the workflow's pinned example exactly.


## 5. Example payloads

Four scenarios covering both extractor modes:

1. **baseline_two_accounts_new_payment** — the workflow's own pinned example (EXTRACT mode).
2. **first_contact_all_accounts** — fresh conversation, "all accounts" + new payment/term (EXTRACT mode).
3. **update_term_only** — only the term changes; accounts/payment/situation carried forward.
4. **clarify_high_installment** — a plan was already offered, customer reacts vaguely with no number (CLARIFY mode).

In [6]:
EXAMPLES: dict[str, AdvisorWebhookPayload] = {
    "baseline_two_accounts_new_payment": baseline_payload,
    "first_contact_all_accounts": AdvisorWebhookPayload(
        userMessage=(
            "ผมเพิ่งตกงานครับ รายได้หายไปเลย อยากขอปรับลดยอดผ่อนทุกบัญชีให้เหลือ"
            "เดือนละ 2000 บาท ผ่อนให้จบภายใน 5 ปี"
        ),
        consultAcc="",
        DebtSituation=None,
        maxPayment=None,
        maxTerm=None,
        narrative="",
        lastAIMessageContent="",
    ),
    "update_term_only": AdvisorWebhookPayload(
        userMessage="อยากผ่อนให้จบภายใน 2 ปีครับ",
        consultAcc="1xxxxxxx0974,1xxxxxxx0975",
        DebtSituation="DebtBurden",
        maxPayment=3000,
        maxTerm=360,
        narrative="Customer already selected both accounts and a 3,000 THB/month budget.",
        lastAIMessageContent="รับทราบค่ะ กำลังประมวลผลแผนให้อยู่ค่ะ",
    ),
    "clarify_high_installment": AdvisorWebhookPayload(
        userMessage="แผนที่เสนอผ่อนแพงไปค่ะ ขอลดลงได้ไหม",
        consultAcc="1xxxxxxx0974,1xxxxxxx0975",
        DebtSituation="DebtBurden",
        maxPayment=5000,
        maxTerm=36,
        narrative="Advisor offered PLAN-001 (5,000 THB/month, 36 months); customer finds it too expensive.",
        offerSoln=[{"planId": "PLAN-001", "term": 36, "installment": 5000}],
        lastAIMessageContent="น้องฟินขอเสนอแผนหมายเลข PLAN-001 ผ่อนชำระ 5,000 บาทต่อเดือน เป็นเวลา 36 เดือนค่ะ",
    ),
}

list(EXAMPLES.keys())

['baseline_two_accounts_new_payment',
 'first_contact_all_accounts',
 'update_term_only',
 'clarify_high_installment']

## 6. Run a single example

Try one payload first to confirm connectivity before looping over all of them.

In [7]:
sample = EXAMPLES["baseline_two_accounts_new_payment"]
print(json.dumps(sample.to_dict(), ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_advisor(sample)
# print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "consultAcc": "",
  "DebtSituation": "DebtBurden",
  "maxPayment": 5200,
  "maxTerm": 360,
  "refPlanID": null,
  "maxPaymentY2": null,
  "maxPaymentY3": null,
  "narrative": "The customer specifies their payment ability, stating they can only afford 3,000 THB per month for their two personal loan accounts combined. They are asking for a new repayment plan based on this amount. Routing to the advisor to create a personalized solution.",
  "accText": "[{\"account_number\":\"1xxxxxxx0974\",\"port\":\"สินเชื่อส่วนบุคคล\",\"creditlimit\":30000,\"os\":19913.56,\"installment\":1700,\"remaining_term\":14},{\"account_number\":\"1xxxxxxx0975\",\"port\":\"สินเชื่อส่วนบุคคล\",\"creditlimit\":80000,\"os\":72742.01,\"installment\":3500,\"remaining_term\":27}]",
  "offerText": "[]",
  "accInfotoExtr": [
    {
      "account_number": "1xxxxxxx0974",
      "port": "สินเชื่อส่วนบุคคล",
      "creditlimit": 30000,
      "os": 19913.56,
      "installment": 1700,
      "remaining_term": 14
    },
   

## 7. Run all examples and inspect results

In [8]:
EXPECTED_MODE = {
    "baseline_two_accounts_new_payment": "EXTRACT",
    "first_contact_all_accounts": "EXTRACT",
    "update_term_only": "EXTRACT",
    "clarify_high_installment": "CLARIFY",
}

rows = []
for label, payload in EXAMPLES.items():
    try:
        result = call_advisor(payload)
        mode = "CLARIFY" if result.get("reConfirmMessage") else "EXTRACT"
        rows.append({
            "case": label,
            "expected_mode": EXPECTED_MODE[label],
            "actual_mode": mode,
            "mode_match": mode == EXPECTED_MODE[label],
            "consultAcc": result.get("consultAcc"),
            "maxPayment": result.get("maxPayment"),
            "maxTerm": result.get("maxTerm"),
            "DebtSituation": result.get("DebtSituation"),
            "reConfirmMessage": result.get("reConfirmMessage"),
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "case": label, "expected_mode": EXPECTED_MODE[label], "actual_mode": None,
            "mode_match": False, "consultAcc": None, "maxPayment": None, "maxTerm": None,
            "DebtSituation": None, "reConfirmMessage": None, "error": str(exc),
        })

results_df = pd.DataFrame(rows)
results_df

,case,expected_mode,actual_mode,mode_match,consultAcc,maxPayment,maxTerm,DebtSituation,reConfirmMessage,error
0,baseline_two_accounts_new_payment,EXTRACT,EXTRACT,True,"1xxxxxxx0974,1xxxxxxx0975",3000,360,DebtBurden,,None
1,first_contact_all_accounts,EXTRACT,EXTRACT,True,"1xxxxxxx0974,1xxxxxxx0975",2000,60,CareerChange,,None
2,update_term_only,EXTRACT,EXTRACT,True,"1xxxxxxx0974,1xxxxxxx0975",3000,24,DebtBurden,,None
3,clarify_high_installment,CLARIFY,CLARIFY,True,"1xxxxxxx0974,1xxxxxxx0975",5000,36,DebtBurden,CLARIFY_HIGH_INSTALLMENT,None


## Notes

- This workflow is **Active**, so the production URL form is
  `{base_url}/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4`.
- This contract is **specific to the webhook path** as it exists today. If
  the workflow is later fixed so the webhook replicates `parsing input for
  extractor`'s flattening (recommended, so both entry points accept the
  same natural payload), this script will need to revert to the nested
  `sessionId`/`conversationDesc`/`userInfo`/`accInfo` shape instead.
- The webhook only returns the **raw extractor output** — to test the full
  advisor pipeline (offer-engine call, clarify reply text, "please specify
  accounts" auto-reply), this workflow needs to be invoked as a
  *sub-workflow* (via n8n's "Execute Workflow" node), not over plain HTTP.
- If your n8n webhook requires authentication, add `auth=`/`headers=` to
  the `requests.post` call inside `call_advisor`.